#Gerando sequências a partir de textos
Modelos de Classificação de Documentos mais elaborados, atualmente, utilizam sequências para realizar as predições. As sequências, diferentemente do modelo Bag-Of-Words, preservam a ordem das palavras durante a análise do conteúdo de um documento e são frequentemente utilizadas com Embeddings, Redes Profundas e Transformers.

No código a seguir, vamos utilizar a biblioteca [Keras](https://keras.io) para realizar a gerar sequências a partir de textos.

In [1]:
from keras.layers import TextVectorization

corpus = ['O cachorro late e pula.',
          'O gato mia e pula.',
          'O cachorro não mia.']

vectorizer_layer = TextVectorization()         # (1)
vectorizer_layer.adapt(corpus)                 # (2)
print(vectorizer_layer(corpus))                # (3)
print(vectorizer_layer.get_vocabulary())


tf.Tensor(
[[2 6 8 5 3]
 [2 9 4 5 3]
 [2 6 7 4 0]], shape=(3, 5), dtype=int64)
['', '[UNK]', np.str_('o'), np.str_('pula'), np.str_('mia'), np.str_('e'), np.str_('cachorro'), np.str_('não'), np.str_('late'), np.str_('gato')]


In [ ]:
# Teste com indices do UNK (unknown)
print(vectorizer_layer(['O gato e o homem são amigos']))   

tf.Tensor([[2 9 5 2 1 1 1]], shape=(1, 7), dtype=int64)


A classe [TextVectorization](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) define estratégias para gerar sequências a partir de textos. Para gerar as sequências a partir dos textos é necessário seguir os seguintes passos:
1. Instanciar a classe;
2. Gerar o dicionário de termos a partir de um córpus (nessa etapa é gerado um dicionário de termos, de forma similar às classes Vectorizer do sklearn);
3. Gerar sequências a partir de textos.

As sequências são basicamente vetores de inteiros. As sequências são geradas com base no dicionário de termos gerado na chamando a instância da classe TextVectorization (a instância é um objeto callable). Cada palavra do córpus é representada como um inteiro, índice do vetor de vocabulário (vectorizer_layer.get_vocabulary) que representa o dicionário de termos. A classe utiliza esse dicionário de termos para construir as sequências.

Um outro ponto importante que vale ser mencionado é o fato de que as sequências foram todas geradas com o mesmo tamanho. Sendo que a classe TextVectorization também se encarregou de garantir isso, removendo palavras do final das frases ou inserindo o índice 0 para completar as sequências que não tivessem o tamanho máximo definido.

O construtor da classe TextVectorization pode receber diferentes argumentos para configurar esse processo, como: número máximo de palavras que devem aparecer nas sequências, opções de filtragem de caractéres, descapitalização dos caractéres, número máximo das sequências geradas (padding_sequences), entre outras.

A seguir, temos outros exemplos de uso dessas estratégias:

In [3]:
corpus_2 = ['O cachorro e o gato estão pulando e deitando.',
            'O cachorro nada.',
            'O gato voa.']
vectorizer_layer = TextVectorization()
vectorizer_layer.adapt(corpus_2)
print(vectorizer_layer(corpus_2))
print(vectorizer_layer.get_vocabulary())
print()

vectorizer_layer = TextVectorization(
    max_tokens=8, output_sequence_length=5)
vectorizer_layer.adapt(corpus_2)
print(vectorizer_layer(corpus_2))
print(vectorizer_layer.get_vocabulary())

tf.Tensor(
[[ 2  5  4  2  3  9  7  4 10]
 [ 2  5  8  0  0  0  0  0  0]
 [ 2  3  6  0  0  0  0  0  0]], shape=(3, 9), dtype=int64)
['', '[UNK]', np.str_('o'), np.str_('gato'), np.str_('e'), np.str_('cachorro'), np.str_('voa'), np.str_('pulando'), np.str_('nada'), np.str_('estão'), np.str_('deitando')]

tf.Tensor(
[[2 5 4 2 3]
 [2 5 1 0 0]
 [2 3 6 0 0]], shape=(3, 5), dtype=int64)
['', '[UNK]', np.str_('o'), np.str_('gato'), np.str_('e'), np.str_('cachorro'), np.str_('voa'), np.str_('pulando')]


Nesse últimos exemplo, nós executamos a mesma camada de vetorização de textos. No entanto, na primeira vez, nós executamos sem passar nenhum parâmetro ao construtor. Na primeira execução, observem que as sequências possuem 9 tokens, 1 para cada palavra das frases. Observem que a segunda e terceira sequência apresentam vários números 0, identificando um espaço em branco (*padding*) deixado para que todas as sequências tenham o mesmo tamanho. O primeiro vetorizador também possui um vocabulário de 10 palavras, com todas as palavras identificadas no córpus.

Na segunda execução, por sua vez, observem que as sequências possuem 5 tokens, dada a definição do parâmetro *output_sequence_length* e o parâmetro *pad_to_max_tokens*. O vocabulário também possui apenas 8 termos dado o uso do parâmetro *max_tokens*.

#Classificação de Documentos com Sequências
Para exemplificar o uso das sequências com Aprendizado Supervisionado, vamos utilizar o mesmo dataset de análise de sentimentos de nossas aulas passadas com comentários na plataforma Buscapé.

In [4]:
import io, zipfile, requests, os

# download the dataset
def download (url, filename=''):
  if (os.path.isfile(filename)):
    print('Arquivo já existente no Runtime... Tudo OK')
    return
  response = requests.get(url)
  with open(f'./{filename}', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')


url = "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/buscape.csv"
download(url, filename='buscape.csv')

Download realizado e arquivo extraído no Runtime... Tudo OK


Com o dataset em nosso sistema de arquivos, vamos carregar os dados de teste e treinamento separadamente.

In [6]:
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


df = pd.read_csv('./buscape.csv').dropna()
X = df['review_text']
y = df['polarity']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

(55219,)
(55219,)
(18407,)
(18407,)


Em seguida, vamos configurar uma Rede Neural com o Keras para predizer a polaridade dos comentários e treinar o modelo. Nessa rede, vamos utilizar apenas uma camada oculta e não vamos fazer uso de Embeddings.

In [8]:
from keras.layers import Dense
from keras.models import Sequential

VOCAB_SIZE = 20000
MAX_SEQUENCE_SIZE = 150
NEURONS = 30
EPOCHS = 5

vectorization_layer = TextVectorization(
    VOCAB_SIZE, output_sequence_length=MAX_SEQUENCE_SIZE)
vectorization_layer.adapt(X_train)

model = Sequential()
model.add(vectorization_layer)
model.add(Dense(NEURONS, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          epochs=EPOCHS, validation_split=0.1)

model.summary()

THRESHOLD=0.5

y_pred = model.predict(X_test)
y_pred_classes = [ 0 if pred < THRESHOLD else 1 for pred in y_pred]
print(classification_report(y_test, y_pred_classes))

Epoch 1/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 2s 974us/step - accuracy: 0.8094 - loss: 67.8989 - val_accuracy: 0.8249 - val_loss: 14.6848
Epoch 2/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1s 919us/step - accuracy: 0.8517 - loss: 4.4360 - val_accuracy: 0.8877 - val_loss: 0.9578
Epoch 3/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1s 909us/step - accuracy: 0.8779 - loss: 0.6164 - val_accuracy: 0.8055 - val_loss: 0.7475
Epoch 4/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1s 905us/step - accuracy: 0.8872 - loss: 0.4749 - val_accuracy: 0.8767 - val_loss: 0.4529
Epoch 5/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1s 916us/step - accuracy: 0.8947 - loss: 0.4028 - val_accuracy: 0.8977 - val_loss: 0.3589


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_4            │ (None, 150)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 30)             │         4,530 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,685 (53.46 KB)

 Trainable params: 4,561 (17.82 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 9,124 (35.64 KB)

576/576 ━━━━━━━━━━━━━━━━━━━━ 0s 604us/step
              precision    recall  f1-score   support

         0.0       0.15      0.01      0.03      1705
         1.0       0.91      0.99      0.95     16702

    accuracy                           0.90     18407
   macro avg       0.53      0.50      0.49     18407
weighted avg       0.84      0.90      0.86     18407



Com o modelo definido e treinado, vamos avaliar sua acurácia com os dados de teste. Em nossos últimos experimentos com esse dataset, nós tivémos os seguintes resultados:
*   O algoritmo que gerou o melhor resultado geral foi o RandomForest;
*   Com o uso de Feature Selection, Scaling, ngrams, GridSearch, tivémos:
  * Acurácia de 0,94;
  * F-Score de 0,62 para a classe 0 e 0,97 para a classe 1.

Vamos utilizar esses valores como referência ao comparar os nossos resultados com classificadores com base em sequências.


Pelos resultados obtidos, vocês podem observar que a acurácia do nosso modelo que utiliza sequências e Redes Neurais com uma camada oculta é bem menor que os modelos que elaboramos anteriormente com o BOW. Especificamente, os resultados foram menores principalmente na identificação de comentários de polaridade negativa (classe 0).

Vale destacar que apesar de ser uma Rede Neural "rasa" (com uma camada oculta), os resultados ainda podem ser melhorados com buscas por melhoria de configurações dos Hyper Parâmetros e aumentando o número de épocas no treinamento. Vocês podem executar alterações nesse código para testar essas possibilidades.

Um outro ponto a ser destacado é o tempo necessário para execução do treinamento. Cada época levou no máximo 10s. Isso é um resultado da configuração da nossa rede e do tamanho do dataset utilizado, sendo que redes mais complexas podem exigir maior tempo de treinamento.

A seguir, vamos explorar outras abordagens que podemos utilizar, nesse processo de classificação.

##Sequências e a Camada de Embeddings
As sequências podem ser utilizadas juntamente com preditores para analisar a ordem de palavras no processo de classificação. No Aprendizado Profundo, as sequências são utilizadas como entrada e passadas para uma camada de Embeddings na Rede Neural.

Para ilustrar como isso funcionaria, vamos considerar o corpus_2 e a camada de vetorização (*vectorization_layer*) que utilizamos anteriormente. Vamos propor o uso de vetores pré-treinados com 2 dimensões (matriz weights) e utilizar esses vetores para inicializar os pesos da nossa camada de Embeddings.

A camada de Embeddings recebe 2 parâmetros obrigatórios: *input_dim* que é o tamanho do vocabulário do vetorizador acrescentando 1 para termos que estejam fora do vocabulário; *output_dim* que é a quantidade de dimensões dos vetores. O terceiro parâmetro passado é uma matriz com valores inicializados de pesos da camada.

O código a seguir implementa uma Rede Neural com essas características.

In [9]:
import numpy as np

from keras.models import Sequential
from keras.layers import Embedding, Reshape

print(corpus_2)              # corpus
weights = np.ones((9, 2))    # (embeddings dimension, vocabulary size)

vocab = vectorizer_layer.get_vocabulary()
for i in range(8):
  weights[i, :] = i
  print(f'{vocab[i]:10}: {weights[i, :]}')


model = Sequential()
model.add(vectorizer_layer)
model.add(Embedding(8 + 1, 2, weights=[weights]))

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print(vectorizer_layer(corpus_2))
result = model.predict(np.array(corpus_2, dtype='object'))

for ind, row in enumerate(result):
  print(f'Doc {ind}: {[ i for i in row[:, 0] ]}')
  print(f'       {[ i for i in row[:, 1] ]}')

model.summary()

['O cachorro e o gato estão pulando e deitando.', 'O cachorro nada.', 'O gato voa.']
          : [0. 0.]
[UNK]     : [1. 1.]
o         : [2. 2.]
gato      : [3. 3.]
e         : [4. 4.]
cachorro  : [5. 5.]
voa       : [6. 6.]
pulando   : [7. 7.]
tf.Tensor(
[[2 5 4 2 3]
 [2 5 1 0 0]
 [2 3 6 0 0]], shape=(3, 5), dtype=int64)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Doc 0: [np.float32(2.0), np.float32(5.0), np.float32(4.0), np.float32(2.0), np.float32(3.0)]
       [np.float32(2.0), np.float32(5.0), np.float32(4.0), np.float32(2.0), np.float32(3.0)]
Doc 1: [np.float32(2.0), np.float32(5.0), np.float32(1.0), np.float32(0.0), np.float32(0.0)]
       [np.float32(2.0), np.float32(5.0), np.float32(1.0), np.float32(0.0), np.float32(0.0)]
Doc 2: [np.float32(2.0), np.float32(3.0), np.float32(6.0), np.float32(0.0), np.float32(0.0)]
       [np.float32(2.0), np.float32(3.0), np.float32(6.0), np.float32(0.0), np.float32(0.0)]


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (3, 5)                 │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (3, 5, 2)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18 (72.00 B)

 Trainable params: 18 (72.00 B)

 Non-trainable params: 0 (0.00 B)

Com a execução do código, nós podemos observar que a camada de Embeddings basicamente substitui os IDs dos termos do vocabulário do vetorizador por seus vetores representantes nas sequências.

Dessa forma, a entrada da camada de Embeddings é uma sequência de tamanho 5 (determinada pelo tamanho máximo das sequências no vetorizador) com IDs referentes às palavras/tokens do vocabulário. A sua saída, por sua vez, é uma sequência de tamanho 5, mas com vetores que representam às palavras/tokens no espaço vetorial definido pelas Embeddings.

No processo de Deep Learning, essa saída é dada como entrada para outras camadas dentro de uma Rede Neural.

##Aprendizado profundo e Embeddings
As embeddings de palavras desempenham um papel fundamental na melhoria das abordagens de classificação de texto. Essas representações numéricas de palavras permitem capturar significados semânticos e relações entre palavras em um espaço vetorial contínuo. Ao utilizar embeddings em modelos de classificação de texto, é possível obter benefícios significativos:
* Captura de Significados Semânticos
* Generalização e Similaridade
* Melhoria na Precisão da Classificação.



In [10]:
import tensorflow as tf


EMBEDDING_DIM = 50

vectorization_layer = TextVectorization(
    VOCAB_SIZE, output_sequence_length=MAX_SEQUENCE_SIZE)
vectorization_layer.adapt(X_train)

model = Sequential()
model.add(vectorization_layer)
model.add(Embedding(VOCAB_SIZE, EMBEDDING_DIM))
model.add(Reshape((MAX_SEQUENCE_SIZE * EMBEDDING_DIM,)))
model.add(Dense(NEURONS, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.build(input_shape=(None, 1))
model.summary()

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          epochs=EPOCHS, validation_split=0.1)


y_pred = model.predict(X_test)
y_pred_classes = [ 0 if pred < THRESHOLD else 1 for pred in y_pred]
print(classification_report(y_test, y_pred_classes))

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_5            │ (None, 150)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 150, 50)        │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7500)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 30)             │       225,030 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,225,061 (4.67 MB)

 Trainable params: 1,225,061 (4.67 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9397 - loss: 0.1813 - val_accuracy: 0.9459 - val_loss: 0.1664
Epoch 2/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9691 - loss: 0.1015 - val_accuracy: 0.9393 - val_loss: 0.2035
Epoch 3/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9875 - loss: 0.0446 - val_accuracy: 0.9372 - val_loss: 0.2712
Epoch 4/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9951 - loss: 0.0195 - val_accuracy: 0.9363 - val_loss: 0.3504
Epoch 5/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9970 - loss: 0.0120 - val_accuracy: 0.9290 - val_loss: 0.3961
576/576 ━━━━━━━━━━━━━━━━━━━━ 1s 974us/step
              precision    recall  f1-score   support

         0.0       0.64      0.55      0.59      1705
         1.0       0.95      0.97      0.96     16702

    accuracy                           0.93     18407
   macro avg       0.80      0.76      0.78     18407
weighted avg       0.93      0.93      0

Nesse modelo, nós incluímos duas camadas em nossa Rede Neural: uma camade de Embedding (similar à camada de embeddings que estudamos anteriormente) e uma camada de Reshape.

A camada de Embeddings aumenta acrescenta uma dimensão na saída relacionada ao número de dimensões que utilizamos para representar cada palavra do nosso modelo. A camada Reshape foi incluída para reduzir essa dimensão, para que os dados pudessem ser passados para a camada Dense que só aceita entradas em duas dimensões.

Vocês podem observar que, com a camada de Embeddings, o nosso classificador obteve resultados de acurácia bem maiores. Mas, dada a maior complexidade da nossa rede, o tempo de treinamento também foi maior, sendo que cada época levou entre 30-40s.

##Transfer Learning com Embeddings

A linha de Aprendizado Supervisionado frequentemente tem utilizado técnicas de *Transfer Learning* para melhorar a acurácia dos modelos.

O *Transfer Learning* é uma técnica em aprendizado de máquina que utiliza o conhecimento aprendido em uma tarefa para melhorar o desempenho em uma tarefa relacionada. Isso é feito reutilizando um modelo pré-treinado em uma tarefa específica e ajustando-o para uma nova tarefa que é similar ou relacionada à primeira.

Dada a camada de Embeddings da nossa Rede Neural, nós poderíamos utilizar uma camada de Embeddings Pré-Treinada para tentar melhorar a acurácia do nosso modelo. Como estamos trabalhando com a Língua Portuguesa, nós podemos ajustar a nossa rede para incorporar Embeddings Pré-Treinados da página do NILC, de forma similar ao que realizamos na aula passada.

Segue um código para fazer o download do modelo de Embeddings Pré-Treinados com Skipgrams.

In [12]:
# download the embeddings
def download_zip (url):
  if (os.path.isfile('./skip_50d.txt')):
    print('Arquivo já existente no Runtime... Tudo OK')
  else:
    response = requests.get(url)

    if response.status_code == 200:
        with zipfile.ZipFile(io.BytesIO(response.content), 'r') as zip_ref:
            zip_ref.extractall('./')
            print("Download concluído e arquivo pronto...")
    else:
        print("Failed to download the zip file.")


zip_url = "http://143.107.183.175:22980/download.php?file=embeddings/word2vec/skip_s50.zip"
download_zip (zip_url)

Arquivo já existente no Runtime... Tudo OK


Com as Embeddings carregadas em nosso sistema de arquivos. Nós podemos utilizá-las com a biblioteca Gensim e, então, passar os vetores das palavas para a camada de Embeddings da nossa rede.

A camada de Embeddings da nossa Rede Neura aceita parâmetros de número de neurônios e *weights*. O número de neurônios da rede precisa ser compatível com a quantidade de dimensões dos vetores utilizados. No nosso caso, nós fizemos o download de vetores contruídos com 50 dimensões.

Os *weights* da camada de Embeddings representam os pesos da conexão das entradas da nossa Rede Neural (pela nossa camada de vetorização em sequências) com a camada de Embeddings. Nesse sentido, nós podemos aproveitar as definições de vetores do nosso arquivo de Embeddings para contruir nossa camada de Embeddings, passando como valores de pesos (*weights*) os valores definidos nos vetores das palavras. Vale destacar que a ordem dos vetores conforme o arquivo do NILC pode ser diferente da ordem dos vetores gerados a partir da nossa abordagem de vetorizador.

O código a seguir implementa uma função que gera a matriz de pesos a partir dos vetores carregados com o Gensim.

In [ ]:
import numpy as np

from gensim.models import KeyedVectors


vectors = KeyedVectors.load_word2vec_format('./skip_50d.txt')

def generate_weight_matrix (vectors, dim=50):
  vectors_vocabulary = vectors.key_to_index.keys()
  vocabulary = ['', '[UNK]'] + list(vectors_vocabulary)
  weights_matrix = np.zeros((len(vocabulary), dim))
  for i, word in enumerate(vocabulary):
    if word in vectors_vocabulary:
      weights_matrix[i] = vectors[word]
    else:
      weights_matrix[i] = np.random.rand(dim)
  return (vocabulary, weights_matrix)


(vocabulary, weights_matrix) = generate_weight_matrix(
    vectors, dim=EMBEDDING_DIM)

print(weights_matrix)
print(weights_matrix[0:5])
print(vocabulary[:10])
print(vectorization_layer.get_vocabulary()[:10])



Com a matriz de pesos definida, nós podemos passar esses parâmetros para a camada de Embeddings utilizando como inicialização de pesos, a nossa matriz carregada a partir de vetores pré-treinados.

In [ ]:
vectorization_layer = TextVectorization(
    len(vocabulary), output_sequence_length=MAX_SEQUENCE_SIZE,
    vocabulary=vocabulary)

model = Sequential()
model.add(vectorization_layer)
model.add(Embedding(len(vocabulary), EMBEDDING_DIM, weights=[weights_matrix]))
model.add(Reshape((MAX_SEQUENCE_SIZE * EMBEDDING_DIM,)))
model.add(Dense(NEURONS, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          epochs=EPOCHS, validation_split=0.1)

model.summary()

y_pred = model.predict(X_test)
y_pred_classes = [ 0 if pred < THRESHOLD else 1 for pred in y_pred]
print(classification_report(y_test, y_pred_classes))

##Aprendizado Profundo com LSTM e Redes Convolucionais
Abordagens baseadas em Aprendizado Profundo têm sido frequentemente utilizadas em aplicações de PLN. Nessa seção, vamos explorar aumentar a complexidade de nossas redes, experimentando o uso de LSTM e Redes Convolucionais.

LSTM (*Long Short-Term Memory*) é um tipo de Rede Neural Recorrente (RNN) projetada para lidar com problemas de sequências longas e curtos. Ela é capaz de aprender padrões em sequências de dados e manter essas informações por períodos prolongados, o que é útil para tarefas como reconhecimento de fala, processamento de linguagem natural.

Em nosso primeiro experimento, vamos investigar o seu uso no lugar da camada intermediária do nosso modelo.

In [ ]:
from keras.layers import LSTM

model = Sequential()
model.add(TextVectorization(len(vocabulary),
    output_sequence_length=MAX_SEQUENCE_SIZE, vocabulary=vocabulary))
model.add(Embedding(len(vocabulary), EMBEDDING_DIM, weights=[weights_matrix]))
model.add(LSTM(NEURONS))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          epochs=EPOCHS, validation_split=0.1)

model.summary()

y_pred = model.predict(X_test)
y_pred_classes = [ 0 if pred < THRESHOLD else 1 for pred in y_pred]
print(classification_report(y_test, y_pred_classes))

Com o uso da camada LSTM, nós podemos observar um incremento significativo no tempo de treinamento da nossa rede, relacionada a cada época. Um outro ponto que pode ser observado é que, com a LSTM, não é necessário utilizar a camada de Flatten, pois a LSTM é capaz de lidar com entradas em 3D (tamanho das sequências, tamanho das amostrar e número de dimensões das Embeddings).



Um outro exemplo de uso de Aprendizado profundo pode ser o uso de Redes Convolucionais.

Uma Rede Neural Convolucional (CNN) é um tipo de rede neural artificial projetada para processar e analisar imagens digitais de forma eficaz. Inspirada na organização do córtex visual dos animais, uma CNN utiliza camadas convolucionais, de correção e pooling para extrair características das imagens e realizar tarefas como reconhecimento de padrões, classificação de objetos e segmentação de imagens. Essas redes são capazes de aprender padrões complexos e hierárquicos.

Embora as CNNs tenham sido desenvolvidas originalmente para tarefas de visão computacional, elas também têm sido aplicadas com sucesso em Processamento de Linguagem Natural (PLN).

Se seguir, vamos utilizar uma camada ConvNet para substituir nossa camada de LSTM e verificar os resultados.

In [ ]:
from keras.layers import Conv1D, MaxPooling1D, Flatten

FILTERS = 64
KERNEL_SIZE = 5
POOL_SIZE = 4

model = Sequential()
model.add(TextVectorization(len(vocabulary),
    output_sequence_length=MAX_SEQUENCE_SIZE, vocabulary=vocabulary))
model.add(Embedding(len(vocabulary), EMBEDDING_DIM, weights=[weights_matrix]))
model.add(Conv1D(FILTERS, KERNEL_SIZE))
model.add(MaxPooling1D(pool_size=POOL_SIZE))
model.add(Reshape(((36 * FILTERS),)))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.build(input_shape=(None, 1))
model.summary()

model.fit(X_train, y_train,
          epochs=EPOCHS, validation_split=0.1)

model.summary()

y_pred = model.predict(X_test)
y_pred_classes = [ 0 if pred < THRESHOLD else 1 for pred in y_pred]
print(classification_report(y_test, y_pred_classes))

A camada de ConvNet frequemente vem acompanhada por uma camada de pooling (MaxPooling1D, dada o uso para classificação de textos). A saída dessas duas camadas é representada em 3 dimensões, por isso foi necessário incluir a camada de Flatten novamente na nossa rede.

A seguir, podemos testar a acurácia dessa rede com o grupo de teste.

#Considerações Finais
Nessa aula, nós trabalhamos com o uso de sequências para alimentar Redes Neurais para realizar tarefas de classificação de documentos. Nós evoluímos nossos modelos, acrescentando complexidades relacionadas ao uso de *Embeddings*, *Transfer Learning* (com *Embeddings* Pré-treinados) e Aprendizado Profundo (LSTM e CNN).

Vale destacar que nossos resultados de acurácia atingiram valores superiores aos modelos utilizados até então. No entanto, nós ainda podemos explorar mais recursos de aprendizado, como:
*   Inclusão de mais camadas nas redes;
*   Camadas de Dropout para reduzir chances de *overfitting*;
*   Aumentar o número de épocas de treinamento;
*   Aumentar a dimensionalidade dos *Embeddings* e neurônios utilizados em cada camada;
*   Realizar GridSearch para procurar por Hyper Parâmetros que otimizam a acurácia.

